# Train Graph Transformer for Redundancy Prediction
This notebook mounts Google Drive, sets up the environment, and trains the PyTorch Geometric Graph Transformer on the `training_pairs.csv` dataset.

In [1]:
import os
import sys
from pathlib import Path
from google.colab import drive
import torch

# ==============================================================================
# Step 1: Mount Drive and set environment variable
# ==============================================================================
print("[STEP 1] Mounting Google Drive...")
drive.mount('/content/drive')
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/CCTV-Multiview-Project"
DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])


Mounted at /content/drive


In [2]:
# ==============================================================================
# Step 2: Verify GPU
# ==============================================================================
print("\n[STEP 2] Verifying GPU...")
if not torch.cuda.is_available():
    print("[FAIL] GPU is not available! Please change the runtime type to T4/A100 GPU and restart.")
    sys.exit(1)
print(f"[PASS] GPU detected: {torch.cuda.get_device_name(0)}")



[STEP 2] Verifying GPU...
[PASS] GPU detected: Tesla T4


In [5]:
# ==============================================================================
# Step 3: Setup Environment
# ==============================================================================
repo_dir = "/content/cctv-multiview-summarization"
if not os.path.exists(repo_dir):
    print(f"[INFO] Cloning repository to {repo_dir}...")
    !git clone https://github.com/Gautam-Shah306/cctv-multiview-summarization.git {repo_dir}

os.chdir(repo_dir)
!git fetch origin
!git checkout feature/stage1-object-detection
!git pull origin feature/stage1-object-detection
!pip install -q -r requirements-colab.txt

print("\n[INFO] Installing PyTorch Geometric...")
!pip install -q torch-geometric


remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 2), reused 6 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 724.45 KiB | 10.35 MiB/s, done.
From https://github.com/Gautam-Shah306/cctv-multiview-summarization
   0288d45..73f02a9  feature/stage1-object-detection -> origin/feature/stage1-object-detection
Already on 'feature/stage1-object-detection'
Your branch is behind 'origin/feature/stage1-object-detection' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Gautam-Shah306/cctv-multiview-summarization
 * branch            feature/stage1-object-detection -> FETCH_HEAD
Updating 0288d45..73f02a9
Fast-forward
 data_manifests/training_features_dino.npz | Bin 0 -> 86074 bytes
 data_manifests/training_features_reid.npz | Bin 0 -> 919610 bytes
 data_manifests/training_pairs.csv         | 327 ++++++++++++++++++++++++

In [6]:
# ==============================================================================
# Step 4: Run Training
# ==============================================================================
print("\n[STEP 4] Running Graph Transformer Training...")
!python -m src.train_graph_transformer



[STEP 4] Running Graph Transformer Training...
[INFO] Device: cuda. (Note: User requested Colab GPU. Running in available environment.)
[INFO] Building PyTorch Geometric graph data object...
[INFO] Train Edges (directional): 520 | Val Edges (directional): 132

[INFO] Starting training...
Epoch 010 | Train Loss: 0.6931 | Val Loss: 0.6931 | Val F1: 0.6667 | Val Acc: 0.5000
Epoch 020 | Train Loss: 0.6929 | Val Loss: 0.6930 | Val F1: 0.6667 | Val Acc: 0.5000
Epoch 030 | Train Loss: 0.6926 | Val Loss: 0.6930 | Val F1: 0.0000 | Val Acc: 0.5000
Epoch 040 | Train Loss: 0.6933 | Val Loss: 0.6928 | Val F1: 0.6704 | Val Acc: 0.5530
Epoch 050 | Train Loss: 0.6933 | Val Loss: 0.6923 | Val F1: 0.0000 | Val Acc: 0.5000
Epoch 060 | Train Loss: 0.6916 | Val Loss: 0.6907 | Val F1: 0.7226 | Val Acc: 0.6742
Epoch 070 | Train Loss: 0.6890 | Val Loss: 0.6858 | Val F1: 0.6857 | Val Acc: 0.5833
Epoch 080 | Train Loss: 0.6835 | Val Loss: 0.6855 | Val F1: 0.0822 | Val Acc: 0.4924
Epoch 090 | Train Loss: 0.6780

In [7]:
# ==============================================================================
# Step 5: Verify Saved Model
# ==============================================================================
# NOTE: The training script saves the model to 'models/' in the repo, NOT drive root directly.
# Let's copy it to Drive so it persists after Colab shuts down!
print("\n[STEP 5] Persisting Model to Google Drive...")
local_model = Path("models/graph_transformer.pt")
drive_model = DRIVE_ROOT / "models" / "graph_transformer.pt"
drive_model.parent.mkdir(parents=True, exist_ok=True)

if local_model.exists():
    import shutil
    shutil.copy2(local_model, drive_model)
    size_mb = drive_model.stat().st_size / (1024 * 1024)
    print(f"[PASS] Model successfully copied to Drive: {drive_model} ({size_mb:.2f} MB)")
else:
    print("[FAIL] Model file was not generated by the training script.")



[STEP 5] Persisting Model to Google Drive...
[PASS] Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer.pt (0.31 MB)
